### Practice — KNN on mpg.csv

Same six stages as the Titanic notebook, on your own.

Question: given a car's specifications, was it built in the USA, Europe or Japan?

Target: origin  ·  File: mpg.csv

In [1]:
import pandas as pd
mpg=pd.read_csv("./mpg.csv")

1. Look at the data

Flow: Shape, columns, what is missing.

Run .info() and .shape. Which column has missing values, and how many?

In [2]:
shape=mpg.shape
print("Shape:",shape)

Shape: (398, 9)


In [4]:
mpg.info()

# horsepower is with null 6 null values

<class 'pandas.DataFrame'>
RangeIndex: 398 entries, 0 to 397
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mpg           398 non-null    float64
 1   cylinders     398 non-null    int64  
 2   displacement  398 non-null    float64
 3   horsepower    392 non-null    float64
 4   weight        398 non-null    int64  
 5   acceleration  398 non-null    float64
 6   model_year    398 non-null    int64  
 7   origin        398 non-null    str    
 8   name          398 non-null    str    
dtypes: float64(4), int64(3), str(2)
memory usage: 28.1 KB


In [5]:
print("Cars from different origin:\n",mpg['origin'].value_counts())

Cars from different origin:
 origin
usa       249
japan      79
europe     70
Name: count, dtype: int64


2. Stage 1 — Data Cleaning

Flow: Fix missing values, drop unusable columns.

Two jobs:

horsepower has blanks. Fill them with the median.
Drop name. In one line below, say why it cannot help the model.

In [6]:
mpg['horsepower']=mpg['horsepower'].fillna(mpg['horsepower'].median())

In [7]:
mpg.drop(columns=['name'], inplace=True)

In [9]:
mpg.info()
#ML works only on numeric values, hence we drop the names

<class 'pandas.DataFrame'>
RangeIndex: 398 entries, 0 to 397
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mpg           398 non-null    float64
 1   cylinders     398 non-null    int64  
 2   displacement  398 non-null    float64
 3   horsepower    398 non-null    float64
 4   weight        398 non-null    int64  
 5   acceleration  398 non-null    float64
 6   model_year    398 non-null    int64  
 7   origin        398 non-null    str    
dtypes: float64(4), int64(3), str(1)
memory usage: 25.0 KB


3. Features and Target

Flow: X is everything the model looks at. y is the answer.

In [11]:
y=mpg['origin']
x=mpg.drop(columns=['origin'])

In [12]:
x.info()

<class 'pandas.DataFrame'>
RangeIndex: 398 entries, 0 to 397
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mpg           398 non-null    float64
 1   cylinders     398 non-null    int64  
 2   displacement  398 non-null    float64
 3   horsepower    398 non-null    float64
 4   weight        398 non-null    int64  
 5   acceleration  398 non-null    float64
 6   model_year    398 non-null    int64  
dtypes: float64(4), int64(3)
memory usage: 21.9 KB


In [13]:
y.info()

<class 'pandas.Series'>
RangeIndex: 398 entries, 0 to 397
Series name: origin
Non-Null Count  Dtype
--------------  -----
398 non-null    str  
dtypes: str(1)
memory usage: 3.2 KB


4. Stage 2 — Train/Test Split

Flow: Hide some rows before preparing anything.

Use test_size=0.2, random_state=0, stratify=y.

In [18]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2, stratify=y, random_state=0)

print(x_train)

      mpg  cylinders  displacement  horsepower  weight  acceleration  \
54   35.0          4          72.0        69.0    1613          18.0   
367  28.0          4         112.0        88.0    2605          19.6   
89   15.0          8         318.0       150.0    3777          12.5   
83   28.0          4          98.0        80.0    2164          15.0   
98   16.0          6         250.0       100.0    3278          18.0   
..    ...        ...           ...         ...     ...           ...   
178  23.0          4         120.0        88.0    2957          17.0   
292  18.5          8         360.0       150.0    3940          13.0   
6    14.0          8         454.0       220.0    4354           9.0   
285  17.0          8         305.0       130.0    3840          15.4   
71   19.0          3          70.0        97.0    2330          13.5   

     model_year  
54           71  
367          82  
89           73  
83           72  
98           73  
..          ...  
178      

In [19]:
print(x_test)

      mpg  cylinders  displacement  horsepower  weight  acceleration  \
116  16.0          8         400.0       230.0    4278           9.5   
15   22.0          6         198.0        95.0    2833          15.5   
132  25.0          4         140.0        75.0    2542          17.0   
40   14.0          8         351.0       153.0    4154          13.5   
396  28.0          4         120.0        79.0    2625          18.6   
..    ...        ...           ...         ...     ...           ...   
57   24.0          4         113.0        95.0    2278          15.5   
96   13.0          8         360.0       175.0    3821          11.0   
173  24.0          4         119.0        97.0    2545          17.0   
275  17.0          6         163.0       125.0    3140          13.6   
92   13.0          8         351.0       158.0    4363          13.0   

     model_year  
116          73  
15           70  
132          74  
40           71  
396          82  
..          ...  
57       

In [20]:
print(y_train)

54      japan
367       usa
89        usa
83        usa
98        usa
        ...  
178    europe
292       usa
6         usa
285       usa
71      japan
Name: origin, Length: 318, dtype: str


In [21]:
print(y_test)

116       usa
15        usa
132       usa
40        usa
396       usa
        ...  
57      japan
96        usa
173     japan
275    europe
92        usa
Name: origin, Length: 80, dtype: str


5. Stage 3 — Feature Engineering

Flow: Put every column on the same scale.

No encoding needed here. After dropping name, every feature is already a number, so there is no text column left for OneHotEncoder. That happens in real projects too.

Print the min and max of each feature. Which column has the largest range?

In [24]:
print("Min of each feature:\n", x.describe().loc['min'])
print("Max of each feature:\n", x.describe().loc['max'])

Min of each feature:
 mpg                9.0
cylinders          3.0
displacement      68.0
horsepower        46.0
weight          1613.0
acceleration       8.0
model_year        70.0
Name: min, dtype: float64
Max of each feature:
 mpg               46.6
cylinders          8.0
displacement     455.0
horsepower       230.0
weight          5140.0
acceleration      24.8
model_year        82.0
Name: max, dtype: float64


In [ ]:
print("Range of each feature:\n", x.describe().loc['max']-x.describe().loc['min'])

Range of each feature:
 mpg               37.6
cylinders          5.0
displacement     387.0
horsepower       184.0
weight          3527.0
acceleration      16.8
model_year        12.0
dtype: float64


In [32]:
rangeFeature=x.describe().loc['max']-x.describe().loc['min']
maxIndex=rangeFeature.argmax()

rangeFeature=rangeFeature.reset_index()
print("Feature with highest range:", rangeFeature.loc[maxIndex, 'index'])

Feature with highest range: weight


Now scale. StandardScaler — fit_transform on train, transform on test.

In [33]:
from sklearn.preprocessing import StandardScaler;
scalar=StandardScaler()

x_train_final=scalar.fit_transform(x_train)
x_test_final=scalar.transform(x_test)

In [35]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

In [42]:
model=KNeighborsClassifier()
model.fit(x_train,y_train)

y_predict_unaccurate=model.predict(x_test)
print("Accuracy when not scaled:", accuracy_score(y_predict_unaccurate, y_test)*100)

Accuracy when not scaled: 68.75


In [43]:
model.fit(x_train_final,y_train)

y_predict_accurate=model.predict(x_test_final)
print("Accuracy when scaled:", accuracy_score(y_predict_accurate, y_test)*100)

# when scaled, accuracy increases, by 6 percent

Accuracy when scaled: 75.0


8. Choosing k

Flow: k is a dial. Try a few settings and look.

Run a loop over k = 1, 3, 5, 7, 9, 11, 15, 21 on the scaled data. Print k and its accuracy on each line.

In [41]:
for k in (1, 3, 5, 7, 9, 11, 15, 21):
    m=KNeighborsClassifier(n_neighbors=k)
    m.fit(x_train_final, y_train)
    y_predict=m.predict(x_test_final)

    print(accuracy_score(y_predict, y_test)*100)

73.75
75.0
75.0
75.0
75.0
72.5
73.75
72.5


Its accuracy roughly remains flat at the range given. Answer of accuracy: 75% percent
best k=3,5,7,9